In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
import torchvision.datasets as datasets
from PIL import Image
import os

In [2]:
# Define Image Augmentation for Self-Supervised Learning
transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [4]:
# Load Dataset (CIFAR-10 as an example)
dataset = datasets.CIFAR100(root="./data", train=True, transform=transform, download=True)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=2)


100.0%


In [5]:
# Define a Simple Encoder (Using ResNet-18 backbone)
class Encoder(nn.Module):
    def __init__(self):
        super(Encoder, self).__init__()
        self.backbone = models.resnet18(pretrained=False)
        self.backbone.fc = nn.Identity()  # Remove the classification head

    def forward(self, x):
        return self.backbone(x)

In [6]:
# Define a Simple Projection Head
class ProjectionHead(nn.Module):
    def __init__(self, in_dim=512, out_dim=128):
        super(ProjectionHead, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Linear(256, out_dim)
        )

    def forward(self, x):
        return self.fc(x)

In [17]:
# Contrastive Loss (NT-Xent Loss for SimCLR)
def contrastive_loss(features, temperature=0.5, device="cpu"):
    batch_size = features.shape[0]
    features = nn.functional.normalize(features, dim=1)  # Normalize embeddings
    similarity_matrix = torch.matmul(features, features.T) / temperature
    labels = torch.arange(batch_size, device=device)  # Ensure labels are on the same device
    loss_fn = nn.CrossEntropyLoss()
    loss = loss_fn(similarity_matrix, labels)
    return loss

In [18]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")  # Use Apple Silicon if available, otherwise CPU
print(f"Using device: {device}")

encoder = Encoder().to(device)
projector = ProjectionHead().to(device)
optimizer = optim.Adam(list(encoder.parameters()) + list(projector.parameters()), lr=1e-3)

for epoch in range(10):  # Training for 10 epochs
    total_loss = 0
    for images, _ in dataloader:
        images = images.to(device)
        optimizer.zero_grad()

        features = encoder(images)
        projections = projector(features)
        loss = contrastive_loss(projections, device=device)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

print("Training Complete!")

Using device: mps
Epoch 1, Loss: 2.9938
Epoch 2, Loss: 2.9481


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x111f3eee0>
Traceback (most recent call last):
  File "/Users/nanmanatvaristhanist/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py", line 1618, in __del__
    self._shutdown_workers()
  File "/Users/nanmanatvaristhanist/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py", line 1582, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/co

KeyboardInterrupt: 